# Demo Churn com MLflow — Recife DUG (versão simples)

Pipeline ponta a ponta de MLOps em um único notebook: **dados sintéticos → feature table no Unity Catalog → LightGBM com Optuna e MLflow → registro de modelo com aliases → serving em tempo real → inferência em batch → monitoramento**.

* Público: Recife Databricks User Group. Runtime alvo: ~15 min em compute serverless. Seed fixa = 42 (reprodutível).

In [0]:
# Bibliotecas extras (não pré-instaladas no serverless). MLflow 3.x já vem no runtime — não pinamos versão.
%pip install databricks-feature-engineering optuna lightgbm shap
dbutils.library.restartPython()

In [0]:
# Configuração global: catálogo, schema, nomes de objetos e seed fixa (reprodutibilidade)
SEED = 42

catalog = "workspace"
schema = "recife_dug_simple"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

current_user = spark.sql("SELECT current_user() AS u").first()["u"]

customers_table = f"{catalog}.{schema}.customers"
tickets_table = f"{catalog}.{schema}.service_tickets"
labels_table = f"{catalog}.{schema}.churn_labels"
feature_table_name = f"{catalog}.{schema}.churn_feature_table"
model_name = f"{catalog}.{schema}.churn_model"
predictions_table_name = f"{catalog}.{schema}.churn_predictions"
baseline_table_name = f"{catalog}.{schema}.churn_predictions_baseline"
endpoint_name = "churn-model-recife-dug-simple"
experiment_name = f"/Users/{current_user}/churn_recife_dug_simple"

import numpy as np
import pandas as pd

np.random.seed(SEED)

print("Usuário:", current_user)
print("Schema em uso:", f"{catalog}.{schema}")
print("Experimento MLflow:", experiment_name)
print("Modelo UC:", model_name)

## 1. DADOS SINTÉTICOS
*Talk track: "Tudo começa com dados. Geramos 5 mil clientes sintéticos de telecom, 15 mil tickets de atendimento e uma taxa de churn de ~33% — tudo determinístico, com seed 42."*

In [0]:
# 1a. Tabela `customers`: 5.000 clientes sintéticos de telecom (seed 42)
rng = np.random.default_rng(SEED)
n = 5000

first_names = ["Ana", "Bruno", "Carlos", "Davi", "Elisa", "Felipe", "Gabriela", "Heitor", "Isabela", "João",
               "Karina", "Lucas", "Marina", "Nelson", "Olívia", "Paulo", "Rafael", "Sabrina", "Thiago", "Vitória"]
last_names = ["Almeida", "Barbosa", "Cardoso", "Duarte", "Estrela", "Farias", "Gomes", "Henrique", "Ipiranga",
              "Jardim", "Lima", "Moura", "Nogueira", "Oliveira", "Pereira", "Quintino", "Rocha", "Silva",
              "Teixeira", "Vasconcelos"]

customer_ids = [f"CUST-{i:05d}" for i in range(1, n + 1)]
name = [f"{rng.choice(first_names)} {rng.choice(last_names)}" for _ in range(n)]
gender = rng.choice(["Female", "Male"], size=n)
senior_citizen = (rng.random(n) < 0.16).astype(int)
age = np.where(senior_citizen == 1, rng.integers(65, 85, n), rng.integers(18, 65, n))
partner = rng.choice(["Yes", "No"], size=n)
dependents = np.where(rng.random(n) < 0.30, "Yes", "No")
contract_type = rng.choice(["Month-to-month", "One year", "Two year"], size=n, p=[0.50, 0.25, 0.25])
tenure_months = np.where(contract_type == "Month-to-month", rng.integers(1, 12, n),
                np.where(contract_type == "One year", rng.integers(6, 48, n), rng.integers(12, 72, n))).astype(int)
internet_service = rng.choice(["Fiber optic", "DSL", "No"], size=n, p=[0.44, 0.34, 0.22])
phone_service = np.where(rng.random(n) < 0.90, "Yes", "No")

# 6 serviços opcionais: "No internet service" quando não há internet
def optional(prob):
    return np.where(internet_service == "No", "No internet service",
                    np.where(rng.random(n) < prob, "Yes", "No"))

streaming_tv = optional(0.50); streaming_movies = optional(0.50)
online_security = optional(0.35); online_backup = optional(0.40)
device_protection = optional(0.40); tech_support = optional(0.35)

paperless_billing = np.where(rng.random(n) < 0.60, "Yes", "No")
payment_method = rng.choice(["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"], size=n)

# Preço mensal: base 20 + internet + telefone + por serviço opcional "Yes" + ruído
internet_fee = np.where(internet_service == "Fiber optic", 44.0, np.where(internet_service == "DSL", 25.0, 0.0))
phone_fee = np.where(phone_service == "Yes", 10.0, 0.0)
optional_fees = sum(np.where(s == "Yes", rng.uniform(5, 12, n), 0.0) for s in
                    [streaming_tv, streaming_movies, online_security, online_backup, device_protection, tech_support])
monthly_charges = np.round(20.0 + internet_fee + phone_fee + optional_fees + rng.uniform(-5, 10, n), 2)
total_charges = np.maximum(monthly_charges * tenure_months + rng.uniform(-50, 50, n), monthly_charges).round(2)

# Probabilidade de churn segundo as regras do negócio (teto 0.85)
p = np.where(contract_type == "Month-to-month", 0.25, np.where(contract_type == "One year", 0.05, 0.0))
p += np.where(tenure_months < 6, 0.15, np.where(tenure_months < 12, 0.08, 0.0))
p += np.where(monthly_charges > 80, 0.10, np.where(monthly_charges > 60, 0.05, 0.0))
p += np.where((online_security == "No") & (tech_support == "No"), 0.08, 0.0)
p += np.where(payment_method == "Electronic check", 0.06, 0.0)
p += np.where(internet_service == "Fiber optic", 0.05, 0.0)
churn = np.where(rng.random(n) < np.minimum(p, 0.85), "Yes", "No")

customers_pdf = pd.DataFrame({
    "customer_id": customer_ids, "name": name, "gender": gender, "senior_citizen": senior_citizen, "age": age,
    "partner": partner, "dependents": dependents, "tenure_months": tenure_months, "contract_type": contract_type,
    "monthly_charges": monthly_charges, "total_charges": total_charges, "payment_method": payment_method,
    "internet_service": internet_service, "phone_service": phone_service, "streaming_tv": streaming_tv,
    "streaming_movies": streaming_movies, "online_security": online_security, "online_backup": online_backup,
    "device_protection": device_protection, "tech_support": tech_support,
    "paperless_billing": paperless_billing, "churn": churn,
})

(spark.createDataFrame(customers_pdf).write.format("delta")
 .mode("overwrite").option("overwriteSchema", "true").saveAsTable(customers_table))

# Comentários em português no Unity Catalog
spark.sql(f"ALTER TABLE {customers_table} SET TBLPROPERTIES ('comment' = 'Clientes sintéticos de telecom — demo de churn do Recife DUG (seed 42)')")
for col, cmt in {
    "customer_id": "Identificador único do cliente", "name": "Nome gerado aleatoriamente",
    "gender": "Gênero do cliente", "senior_citizen": "1 se idoso (65+), senão 0", "age": "Idade em anos",
    "partner": "Possui parceiro(a)", "dependents": "Possui dependentes", "tenure_months": "Meses de permanência",
    "contract_type": "Tipo de contrato", "monthly_charges": "Cobrança mensal (USD)",
    "total_charges": "Cobrança acumulada (USD)", "payment_method": "Forma de pagamento",
    "internet_service": "Tipo de internet", "phone_service": "Possui telefone",
    "streaming_tv": "Serviço de TV por streaming", "streaming_movies": "Streaming de filmes",
    "online_security": "Segurança online", "online_backup": "Backup online",
    "device_protection": "Proteção de equipamento", "tech_support": "Suporte técnico",
    "paperless_billing": "Fatura digital (sem papel)", "churn": "Alvo: Yes = cancelou o serviço",
}.items():
    spark.sql(f"ALTER TABLE {customers_table} ALTER COLUMN {col} COMMENT '{cmt}'")

print(f"{customers_table}: {spark.table(customers_table).count():,} linhas gravadas")

In [0]:
# 1b. Tabela `service_tickets`: 15.000 tickets — churners geram 3x mais tickets, com skew para billing/cancellation e prioridade alta
rng = np.random.default_rng(SEED)
m = 15000

cust = spark.table(customers_table).select("customer_id", "churn").toPandas()
churn_map = dict(zip(cust["customer_id"], cust["churn"]))
all_ids = sorted(churn_map.keys())

# Peso 3x para churners na amostragem de quem abre ticket
w = np.array([3.0 if churn_map[c] == "Yes" else 1.0 for c in all_ids])
w = w / w.sum()
ticket_customer = rng.choice(all_ids, size=m, p=w)
is_churner = np.array([churn_map[c] == "Yes" for c in ticket_customer])

categories = ["billing", "cancellation", "technical", "general", "upgrade"]
category = np.where(is_churner,
                    rng.choice(categories, size=m, p=[0.35, 0.25, 0.20, 0.10, 0.10]),
                    rng.choice(categories, size=m, p=[0.15, 0.05, 0.25, 0.35, 0.20]))

priorities = ["low", "medium", "high", "critical"]
priority = np.where(is_churner,
                    rng.choice(priorities, size=m, p=[0.10, 0.30, 0.35, 0.25]),
                    rng.choice(priorities, size=m, p=[0.40, 0.40, 0.15, 0.05]))

status = rng.choice(["open", "resolved", "escalated"], size=m, p=[0.15, 0.65, 0.20])
resolved = status == "resolved"
resolution_time_hours = np.where(resolved, rng.exponential(24.0, m) + 0.5, np.nan)
created_date = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365 * 24 * 3600, m), unit="s")

desc_map = {
    "billing": "Customer disputes a charge on the monthly invoice.",
    "cancellation": "Customer requests cancellation of the service.",
    "technical": "Customer reports intermittent internet connection.",
    "general": "Customer asks general questions about the account.",
    "upgrade": "Customer wants to upgrade the current plan.",
}

tickets_pdf = pd.DataFrame({
    "ticket_id": [f"TKT-{i:06d}" for i in range(1, m + 1)],
    "customer_id": ticket_customer,
    "created_date": created_date,
    "category": category,
    "priority": priority,
    "status": status,
    "resolution_time_hours": resolution_time_hours,
    "description": [desc_map[c] for c in category],
})
# Não resolvidos ficam com NULL (não NaN)
tickets_pdf["resolution_time_hours"] = tickets_pdf["resolution_time_hours"].astype(object).where(tickets_pdf["status"] == "resolved", None)

(spark.createDataFrame(tickets_pdf).write.format("delta")
 .mode("overwrite").option("overwriteSchema", "true").saveAsTable(tickets_table))

spark.sql(f"ALTER TABLE {tickets_table} SET TBLPROPERTIES ('comment' = 'Tickets de atendimento sintéticos — churners geram 3x mais tickets (seed 42)')")
for col, cmt in {
    "ticket_id": "Identificador único do ticket", "customer_id": "FK para customers",
    "created_date": "Data/hora de abertura (2025)", "category": "Categoria do ticket",
    "priority": "Prioridade (low/medium/high/critical)", "status": "Situação (open/resolved/escalated)",
    "resolution_time_hours": "Horas até resolução (NULL se não resolvido)", "description": "Descrição do ticket",
}.items():
    spark.sql(f"ALTER TABLE {tickets_table} ALTER COLUMN {col} COMMENT '{cmt}'")

print(f"{tickets_table}: {spark.table(tickets_table).count():,} linhas gravadas")

In [0]:
# 1c. Tabela `churn_labels`: alvo, data do rótulo (2025-06-01) e split 70/15/15
rng = np.random.default_rng(SEED)

labels_pdf = spark.table(customers_table).select("customer_id", "churn").toPandas()
labels_pdf["label_date"] = pd.Timestamp("2025-06-01")
labels_pdf["split"] = rng.choice(["train", "val", "test"], size=len(labels_pdf), p=[0.70, 0.15, 0.15])

(spark.createDataFrame(labels_pdf).write.format("delta")
 .mode("overwrite").option("overwriteSchema", "true").saveAsTable(labels_table))

spark.sql(f"ALTER TABLE {labels_table} SET TBLPROPERTIES ('comment' = 'Rótulos de churn por cliente: alvo, data do rótulo (2025-06-01) e split treino/validação/teste')")
for col, cmt in {"customer_id": "Identificador único do cliente", "churn": "Alvo: Yes = cancelou",
                 "label_date": "Data do rótulo (2025-06-01)", "split": "train / val / test (70/15/15)"}.items():
    spark.sql(f"ALTER TABLE {labels_table} ALTER COLUMN {col} COMMENT '{cmt}'")

print(f"{labels_table}: {spark.table(labels_table).count():,} linhas gravadas")

In [0]:
# 1d. Verificação: contagens, taxa de churn e divisão de splits
for nm, tbl in [("customers", customers_table), ("service_tickets", tickets_table), ("churn_labels", labels_table)]:
    print(f"{nm:16s} {tbl}  ->  {spark.table(tbl).count():,} linhas")

stats = spark.sql(f"""
SELECT ROUND(100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
       SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churners
FROM {customers_table}""").first()
print(f"\nTaxa de churn: {stats['churn_rate_pct']}% ({stats['churners']:,} de 5.000 clientes)")

display(spark.sql(f"SELECT split, COUNT(*) AS n, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct FROM {labels_table} GROUP BY split ORDER BY split"))

## 2. EXPLORAÇÃO
*Talk track: "Antes de modelar, olhemos os dados: churn de ~32%, concentrado em contratos mensais, tenure curto e cobranças altas — e quem cancela reclama mais."*

In [0]:
%sql
-- 2a. Distribuição do churn (contagem e %)
SELECT churn,
       COUNT(*) AS clientes,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM workspace.recife_dug_simple.customers
GROUP BY churn
ORDER BY churn

In [0]:
%sql
-- 2b. Churn por tipo de contrato
SELECT contract_type,
       COUNT(*) AS clientes,
       SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS cancelaram,
       ROUND(100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_rate_pct
FROM workspace.recife_dug_simple.customers
GROUP BY contract_type
ORDER BY churn_rate_pct DESC

In [0]:
%sql
-- 2c. Médias de tenure e cobranças por status de churn
SELECT churn,
       COUNT(*) AS clientes,
       ROUND(AVG(tenure_months), 1) AS tenure_medio_meses,
       ROUND(AVG(monthly_charges), 2) AS cobranca_mensal_media,
       ROUND(AVG(total_charges), 2) AS cobranca_total_media
FROM workspace.recife_dug_simple.customers
GROUP BY churn
ORDER BY churn

In [0]:
%sql
-- 2d. Tickets por categoria
SELECT category,
       COUNT(*) AS tickets,
       SUM(CASE WHEN status = 'resolved' THEN 1 ELSE 0 END) AS resolvidos,
       ROUND(AVG(resolution_time_hours), 1) AS horas_medias_resolucao
FROM workspace.recife_dug_simple.service_tickets
GROUP BY category
ORDER BY tickets DESC

In [0]:
%sql
-- 2e. Média de tickets por cliente, separada por status de churn (LEFT JOIN inclui quem nunca abriu ticket)
SELECT c.churn,
       COUNT(DISTINCT c.customer_id) AS clientes,
       COUNT(t.ticket_id) AS tickets,
       ROUND(COUNT(t.ticket_id) * 1.0 / COUNT(DISTINCT c.customer_id), 2) AS media_tickets_por_cliente
FROM workspace.recife_dug_simple.customers c
LEFT JOIN workspace.recife_dug_simple.service_tickets t
  ON c.customer_id = t.customer_id
GROUP BY c.churn
ORDER BY c.churn

### Resumo da exploração
* **Contrato Month-to-month, tenure curto e cobranças altas** correlacionam com churn — exatamente as regras de negócio que injetamos nos dados sintéticos.
* **Quem cancela abre ~3x mais tickets**, concentrados em **billing** e **cancellation** e com prioridade high/critical.
* Esses drivers devem reaparecer como os principais fatores do modelo (SHAP, seção 4) e no monitoramento de drift (seção 7).

## 3. FEATURE ENGINEERING COM UNITY CATALOG
*Talk track: "Boas features evitam vazamento de dados: agregamos os tickets apenas até a data do rótulo, registramos tudo como feature table no Unity Catalog e criamos uma função SQL on-demand."

### 3.1 Point-in-time: evitar vazamento de dados
Agregamos apenas tickets com `created_date <= 2025-06-01` (a `label_date`). Se usássemos todos os tickets, o modelo veria reclamações que aconteceram **depois** da decisão de cancelar — informação que não existiria no momento da predição em produção. Isso é *data leakage*: métricas lindas no treino, modelo inútil no mundo real.

In [0]:
# 3a. Agregação point-in-time dos tickets + join com clientes + Pandas UDF para nº de serviços opcionais
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
from pyspark.sql.functions import pandas_udf

label_date = "2025-06-01"

ticket_agg = (spark.table(tickets_table)
    .filter(F.to_date(F.col("created_date")) <= F.lit(label_date))
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("total_tickets"),
        F.sum(F.when(F.col("category") == "billing", 1).otherwise(0)).alias("billing_tickets"),
        F.sum(F.when(F.col("category") == "technical", 1).otherwise(0)).alias("technical_tickets"),
        F.sum(F.when(F.col("category") == "cancellation", 1).otherwise(0)).alias("cancellation_tickets"),
        F.sum(F.when(F.col("priority").isin("high", "critical"), 1).otherwise(0)).alias("high_priority_tickets"),
        F.avg(F.when(F.col("status") == "resolved", F.col("resolution_time_hours"))).alias("avg_resolution_hours"),
        F.max("created_date").alias("last_ticket_date"),
    ))

# Left-join: clientes sem tickets até a data do rótulo ficam com 0
joined = (spark.table(customers_table)
    .join(ticket_agg, "customer_id", "left")
    .fillna(0, subset=["total_tickets", "billing_tickets", "technical_tickets",
                       "cancellation_tickets", "high_priority_tickets"])
    .fillna(0.0, subset=["avg_resolution_hours"]))

# Pandas UDF: quantos dos 6 serviços opcionais o cliente contratou ("Yes")
@pandas_udf(LongType())
def num_optional_services(streaming_tv, streaming_movies, online_security, online_backup, device_protection, tech_support):
    return ((streaming_tv == "Yes").astype(int)
            + (streaming_movies == "Yes").astype(int)
            + (online_security == "Yes").astype(int)
            + (online_backup == "Yes").astype(int)
            + (device_protection == "Yes").astype(int)
            + (tech_support == "Yes").astype(int))

joined = joined.withColumn("num_optional_services",
    num_optional_services("streaming_tv", "streaming_movies", "online_security",
                          "online_backup", "device_protection", "tech_support"))
joined = joined.withColumn("update_timestamp", F.current_timestamp())

feature_cols = [
    "customer_id", "tenure_months", "contract_type", "monthly_charges", "total_charges",
    "payment_method", "internet_service", "phone_service", "paperless_billing",
    "partner", "dependents", "senior_citizen", "num_optional_services",
    "total_tickets", "billing_tickets", "technical_tickets", "cancellation_tickets",
    "high_priority_tickets", "avg_resolution_hours", "update_timestamp",
]
feature_df = joined.select(*feature_cols)

print(f"DataFrame de features: {feature_df.count():,} clientes x {len(feature_df.columns)} colunas")
display(feature_df.limit(5))

In [0]:
# 3b. Cria a feature table no Unity Catalog (idempotente: drop antes de criar)
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

spark.sql(f"DROP TABLE IF EXISTS {feature_table_name}")
fe.create_table(
    name=feature_table_name,
    primary_keys=["customer_id"],
    df=feature_df,
    schema=feature_df.schema,
    description="Features de churn por cliente — perfil contratual + agregações point-in-time de tickets até 2025-06-01",
)

print(f"Feature table criada: {feature_table_name} ({spark.table(feature_table_name).count():,} linhas)")

In [0]:
# 3c. Função Python sob demanda no Unity Catalog: cobrança média por mês de contrato
# (UDF LANGUAGE PYTHON — é o tipo exigido pelo FeatureFunction do databricks-feature-engineering)
spark.sql(f"DROP FUNCTION IF EXISTS {catalog}.{schema}.avg_price_increase")
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.avg_price_increase(monthly_charges DOUBLE, tenure_months BIGINT)
RETURNS DOUBLE
LANGUAGE PYTHON
COMMENT 'Cobrança mensal dividida pelos meses de contrato — sinal de aumento de preço'
AS $$
    if tenure_months > 0:
        return monthly_charges / tenure_months
    return 0.0
$$
""")

# Teste rápido da função
display(spark.sql(f"""
SELECT monthly_charges, tenure_months,
       {catalog}.{schema}.avg_price_increase(monthly_charges, tenure_months) AS avg_price_increase
FROM {customers_table} LIMIT 5"""))

### 3.2 Por que uma feature table (e não um Delta comum)?
* **Reuso**: features compartilhadas entre modelos e times, com descoberta centralizada no Unity Catalog.
* **Sem train/serve skew**: `FeatureLookup` garante exatamente os mesmos valores e cálculo no treino e no serving — a feature não é “reinventada” em outra query.
* **Linhagem e governança**: a feature table aparece na linhagem do Unity Catalog, com chave primária, dono e permissões. Um `saveAsTable` comum não dá nada disso.

## 4. TREINO E REGISTRO DO MODELO
*Talk track: "Agora o MLflow entra: Optuna roda 10 experimentos aninhados, o melhor modelo é registrado no Unity Catalog com aliases Champion e Challenger — os stages deprecados morreram."

In [0]:
# 4a. MLflow: experimento + registry no Unity Catalog
import mlflow

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_name)

exp = mlflow.get_experiment_by_name(experiment_name)
print(f"Experimento: {exp.name} (id {exp.experiment_id})")
print(f"MLflow {mlflow.__version__} | registry: databricks-uc")

In [0]:
# 4b. Training set via Feature Engineering: FeatureLookup + FeatureFunction (função on-demand do UC)
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup, FeatureFunction
from pyspark.sql import functions as F

fe = FeatureEngineeringClient()

labels_df = (spark.table(labels_table)
    .filter(F.col("split").isin("train", "val"))
    .select("customer_id", "churn", "label_date", "split"))

training_set = fe.create_training_set(
    df=labels_df,
    feature_lookups=[
        FeatureLookup(table_name=feature_table_name, lookup_key=["customer_id"]),
        FeatureFunction(
            udf_name=f"{catalog}.{schema}.avg_price_increase",
            output_name="avg_price_increase",
            input_bindings={"monthly_charges": "monthly_charges", "tenure_months": "tenure_months"},
        ),
    ],
    label="churn",
    exclude_columns=["customer_id", "label_date", "update_timestamp"],  # mantemos `split` p/ separar treino/validação
)

training_df = training_set.load_df().toPandas()
print(f"Training set: {training_df.shape[0]:,} linhas x {training_df.shape[1]} colunas")
display(training_df.head(5))

In [0]:
# 4c. Preparação: alvo numérico, split treino/validação e pipeline sklearn + LightGBM
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

le = LabelEncoder()
training_df["target"] = le.fit_transform(training_df["churn"])  # No=0, Yes=1
print("Classes:", dict(zip(le.classes_, le.transform(le.classes_))))

numeric_cols = ["tenure_months", "monthly_charges", "total_charges", "senior_citizen",
                "num_optional_services", "total_tickets", "billing_tickets", "technical_tickets",
                "cancellation_tickets", "high_priority_tickets", "avg_resolution_hours", "avg_price_increase"]
categorical_cols = ["contract_type", "payment_method", "internet_service", "phone_service",
                    "paperless_billing", "partner", "dependents"]
input_cols = numeric_cols + categorical_cols

train_df = training_df[training_df["split"] == "train"]
val_df = training_df[training_df["split"] == "val"]
X_train, y_train = train_df[input_cols], train_df["target"]
X_val, y_val = val_df[input_cols], val_df["target"]
print(f"Treino: {len(train_df):,} | Validação: {len(val_df):,} | Churn no treino: {y_train.mean():.2%}")

def make_pipeline(params):
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ])
    model = LGBMClassifier(random_state=SEED, verbose=-1, **params)
    return Pipeline([("preprocessor", preprocessor), ("model", model)])

In [0]:
# 4d. Optuna: 10 trials, cada um um child run aninhado; parent run "optuna_tuning" loga o melhor resultado
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

with mlflow.start_run(run_name="optuna_tuning") as parent_run:
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
        }
        with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):
            pipe = make_pipeline(params)
            pipe.fit(X_train, y_train)
            preds = pipe.predict(X_val)
            proba = pipe.predict_proba(X_val)[:, 1]
            f1 = f1_score(y_val, preds)
            auc = roc_auc_score(y_val, proba)
            mlflow.log_params(params)
            mlflow.log_metrics({"f1_score": f1, "roc_auc": auc})
        return f1

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=10)

    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_f1_score", study.best_value)

best_params = study.best_params
print("Melhores parâmetros:", best_params)
print(f"Melhor F1 (validação): {study.best_value:.4f}")

In [0]:
# 4e. Run "final_model": treino final, log_model com assinatura, smoke test e mlflow.evaluate
import mlflow.sklearn

with mlflow.start_run(run_name="final_model") as final_run:
    final_run_id = final_run.info.run_id

    final_model = make_pipeline(best_params)
    final_model.fit(X_train, y_train)

    val_pred = final_model.predict(X_val)
    val_proba = final_model.predict_proba(X_val)[:, 1]
    metrics = {
        "f1_score": f1_score(y_val, val_pred),
        "roc_auc": roc_auc_score(y_val, val_proba),
        "precision": precision_score(y_val, val_pred),
        "recall": recall_score(y_val, val_pred),
    }
    mlflow.log_metrics(metrics)
    mlflow.log_params(best_params)
    print("Métricas (validação):", {k: round(v, 4) for k, v in metrics.items()})

    # input_example -> MLflow infere a assinatura automaticamente
    mlflow.sklearn.log_model(
        sk_model=final_model,
        artifact_path="model",
        input_example=X_val.head(5),
    )

    # Smoke test: recarregar direto do run
    reloaded = mlflow.sklearn.load_model(f"runs:/{final_run_id}/model")
    print("Smoke test (10 primeiros):", reloaded.predict(X_val.head(10)).tolist())

    # mlflow.evaluate foi deprecado no MLflow 3 -> mlflow.models.evaluate
    # (a opção log_model_explainability mudou para evaluator_config)
    eval_df = X_val.copy()
    eval_df["churn"] = y_val.values
    try:
        result = mlflow.models.evaluate(
            model=f"runs:/{final_run_id}/model",
            data=eval_df,
            targets="churn",
            model_type="classifier",
            evaluator_config={"log_model_explainability": False},
        )
        print("mlflow.models.evaluate concluído — artefatos no run final_model")
    except Exception as e:
        print("mlflow.models.evaluate indisponível, mantendo métricas manuais:", type(e).__name__, str(e)[:200])

print(f"Run final: {final_run_id}")

In [0]:
# 4f. Registro no Unity Catalog: nova versão + descrição + tags + aliases Champion/Challenger
from mlflow import MlflowClient
import time

client = MlflowClient(registry_uri="databricks-uc")

mv = mlflow.register_model(f"runs:/{final_run_id}/model", model_name, await_registration_for=300)
version = str(mv.version)

client.update_registered_model(
    name=model_name,
    description="Modelo LightGBM de churn — features do Unity Catalog + Optuna (10 trials, F1 na validação)")
client.update_model_version(
    name=model_name, version=version,
    description=f"F1={metrics['f1_score']:.4f} | ROC-AUC={metrics['roc_auc']:.4f} | run {final_run_id}")
client.set_registered_model_tag(model_name, "training_framework", "lightgbm+optuna")
client.set_registered_model_tag(model_name, "f1_score", f"{metrics['f1_score']:.4f}")
client.set_registered_model_tag(model_name, "roc_auc", f"{metrics['roc_auc']:.4f}")

# Aliases substituem os stages deprecados: código downstream carrega models:/<nome>@Champion
for alias in ["Challenger", "Champion"]:
    client.set_registered_model_alias(model_name, alias, version)

champ = client.get_model_version_by_alias(model_name, "Champion")
print(f"Registrado: {model_name} v{champ.version} | aliases: Challenger, Champion")

In [0]:
# 4g. Interpretabilidade: SHAP + matriz de confusão + curva ROC
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve

val_pred = final_model.predict(X_val)
val_proba = final_model.predict_proba(X_val)[:, 1]

preprocessor = final_model.named_steps["preprocessor"]
lgbm_model = final_model.named_steps["model"]
X_val_t = preprocessor.transform(X_val)
feat_names = list(preprocessor.get_feature_names_out())

explainer = shap.TreeExplainer(lgbm_model, data=preprocessor.transform(X_train.head(200)))
sv = explainer.shap_values(X_val_t)

# TreeExplainer pode devolver lista (1 por classe) ou array 2D/3D — normalizamos p/ a classe positiva
if isinstance(sv, list):
    sv_pos = sv[1]
elif sv.ndim == 3:
    sv_pos = sv[:, :, 1]
else:
    sv_pos = sv

plt.figure(figsize=(10, 6))
shap.summary_plot(sv_pos, X_val_t, feature_names=feat_names, show=False)
plt.title("SHAP — impacto das features na predição de churn")
plt.tight_layout()
display(plt.gcf()); plt.close("all")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_val, val_pred), display_labels=["No", "Yes"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Matriz de confusão (validação)")
fpr, tpr, _ = roc_curve(y_val, val_proba)
axes[1].plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set_title("Curva ROC (validação)"); axes[1].legend()
plt.tight_layout()
display(plt.gcf()); plt.close("all")

### 4.1 Leitura dos drivers de churn
1. **Tickets dominam o ranking**: volume de tickets de **alta prioridade**, **cancellation** e **billing** são os preditores mais fortes — reclamar muito (e reclamar de cobrança) é o degrau antes de cancelar.
2. **Contrato Month-to-month e tenure curto** vêm logo atrás — cliente novo sem vínculo é o segundo grupo de risco.
3. **Cobranças**: `monthly_charges` e `avg_price_increase` completam o quadro — preço alto por mês de contrato pesa na decisão de sair.

## 5. MODEL SERVING EM TEMPO REAL
*Talk track: "Batch cobre o dia a dia; serving em tempo real cobre o agora: a aplicação manda o vetor de features e recebe o score de churn em milissegundos."*

In [0]:
# 5a. Endpoint do Champion: cria (ou atualiza) o endpoint via SDK; se serving não estiver disponível, cai no fallback do payload/curl
import json
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput, ServedEntityInput, ServingModelWorkloadType,
    EndpointStateReady, EndpointStateConfigUpdate,
)
from mlflow import MlflowClient

w = WorkspaceClient()
client = MlflowClient(registry_uri="databricks-uc")

champ = client.get_model_version_by_alias(model_name, "Champion")
champion_version = str(champ.version)
print(f"Champion: {model_name} v{champion_version}")

# Payload de teste: 5 clientes da feature table (sem customer_id/update_timestamp) + avg_price_increase via função do UC
sample = spark.sql(f"""
SELECT f.customer_id, f.tenure_months, f.contract_type, f.monthly_charges, f.total_charges,
       f.payment_method, f.internet_service, f.phone_service, f.paperless_billing, f.partner,
       f.dependents, f.senior_citizen, f.num_optional_services, f.total_tickets, f.billing_tickets,
       f.technical_tickets, f.cancellation_tickets, f.high_priority_tickets, f.avg_resolution_hours,
       {catalog}.{schema}.avg_price_increase(f.monthly_charges, f.tenure_months) AS avg_price_increase
FROM {feature_table_name} f LIMIT 5
""").toPandas()
sample_ids = sample["customer_id"].tolist()
payload_records = sample.drop(columns=["customer_id"]).to_dict(orient="records")
payload = {"dataframe_records": payload_records}

endpoint_ready = False
state = None
try:
    served_entity = ServedEntityInput(
        entity_name=model_name,
        entity_version=champion_version,
        workload_type=ServingModelWorkloadType.CPU,
        workload_size="Small",
        scale_to_zero_enabled=True,
    )
    try:
        w.serving_endpoints.get(name=endpoint_name)
        print("Endpoint já existe — atualizando para o Champion atual...")
        w.serving_endpoints.update_config(name=endpoint_name, served_entities=[served_entity])
    except NotFound:
        print(f"Criando endpoint {endpoint_name} (modelo v{champion_version}, Small, scale-to-zero)...")
        w.serving_endpoints.create(name=endpoint_name, config=EndpointCoreConfigInput(name=endpoint_name, served_entities=[served_entity]))

    # Aguarda ficar READY (até 45 min)
    deadline = time.time() + 45 * 60
    while time.time() < deadline:
        state = w.serving_endpoints.get(name=endpoint_name).state
        if state.ready == EndpointStateReady.READY and state.config_update == EndpointStateConfigUpdate.NOT_UPDATING:
            endpoint_ready = True
            break
        if state.config_update in (EndpointStateConfigUpdate.UPDATE_FAILED, EndpointStateConfigUpdate.UPDATE_CANCELED):
            raise RuntimeError(f"Deploy falhou: {state.config_update}")
        time.sleep(15)
    print(f"Endpoint {endpoint_name}: {'PRONTO' if endpoint_ready else 'não ficou pronto no prazo'} (state={state.ready})")
except Exception as e:
    print(f"⚠️ Model Serving indisponível/limitado nesta workspace do Free Trial: {type(e).__name__}: {str(e)[:300]}")
    print("Fallback: a célula seguinte mostra o payload e o curl equivalentes para a apresentação.")

In [0]:
# 5b. Chama o endpoint com dataframe_records e imprime a predição por cliente — ou mostra payload/curl equivalentes
if endpoint_ready:
    try:
        resp = w.serving_endpoints.query(name=endpoint_name, dataframe_records=payload_records)
        for cid, pred in zip(sample_ids, resp.predictions):
            print(f"{cid} -> {pred}")
    except Exception as e:
        print(f"Consulta falhou: {type(e).__name__}: {str(e)[:300]}")
        print("Resposta bruta seria via POST /serving-endpoints/%s/invocations" % endpoint_name)
else:
    print("=== Payload equivalente (POST /serving-endpoints/%s/invocations) ===" % endpoint_name)
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:2500])
    print("\n=== curl equivalente ===")
    print(f'''curl -X POST \\
  "{w.config.host}/serving-endpoints/{endpoint_name}/invocations" \\
  -H "Authorization: Bearer $TOKEN" \\
  -H "Content-Type: application/json" \\
  -d '{json.dumps(payload)}' '''.replace("\\\\", "\\"))

### Tempo real vs. batch
* **Tempo real**: a aplicação envia o vetor de features de **um** cliente e recebe a probabilidade de churn na hora — decisão no contato (ex.: oferta de retenção durante o atendimento).
* **Batch** (seção 6): recarregamos o modelo `@Champion` e pontuamos **milhares** de clientes de uma vez — listas para campanhas, CRM e o monitoramento.
* Mesmo modelo, mesma versão, dois padrões de consumo — e a linha de comando muda só de `serving_endpoints.query` para `mlflow.sklearn.load_model`.

## 6. INFERÊNCIA EM BATCH
*Talk track: "Agora o outro lado: carregamos o Champion pela alias — sem pinar número de versão — e pontuamos todo o conjunto de teste de uma vez, gravando as predições no Unity Catalog."

In [0]:
# 6a. Batch: split test -> join com features + avg_price_increase -> modelo @Champion -> tabela churn_predictions
import mlflow.sklearn

# Matriz do split test (mesmas colunas do treino), mantendo o rótulo p/ uso no monitoramento
batch_pdf = spark.sql(f"""
SELECT c.customer_id, f.tenure_months, f.contract_type, f.monthly_charges, f.total_charges,
       f.payment_method, f.internet_service, f.phone_service, f.paperless_billing, f.partner,
       f.dependents, f.senior_citizen, f.num_optional_services, f.total_tickets, f.billing_tickets,
       f.technical_tickets, f.cancellation_tickets, f.high_priority_tickets, f.avg_resolution_hours,
       {catalog}.{schema}.avg_price_increase(f.monthly_charges, f.tenure_months) AS avg_price_increase,
       c.churn
FROM {labels_table} c
JOIN {feature_table_name} f ON c.customer_id = f.customer_id
WHERE c.split = 'test'
""").toPandas()

X_batch = batch_pdf[input_cols]

# Carrega o Champion pela alias — quem aponta a alias controla qual versão roda
batch_model = mlflow.sklearn.load_model(f"models:/{model_name}@Champion")

batch_pdf["prediction"] = batch_model.predict(X_batch).astype("float64")
batch_pdf["churn_probability"] = batch_model.predict_proba(X_batch)[:, 1]

# customer_id + todas as colunas de features + prediction + churn_probability
pred_cols = ["customer_id"] + input_cols + ["prediction", "churn_probability"]
predictions_pdf = batch_pdf[pred_cols]

(spark.createDataFrame(predictions_pdf).write.format("delta")
 .mode("overwrite").option("overwriteSchema", "true").saveAsTable(predictions_table_name))

spark.sql(f"ALTER TABLE {predictions_table_name} SET TBLPROPERTIES ('comment' = 'Predições de churn do split test — modelo Champion (LightGBM)')")

print(f"{predictions_table_name}: {spark.table(predictions_table_name).count():,} linhas gravadas")
print("\nTop 10 clientes de maior risco:")
display(predictions_pdf.sort_values("churn_probability", ascending=False).head(10))

## 7. LAKEHOUSE MONITORING
*Talk track: "Modelo em produção não é 'deploy e esqueça': o Lakehouse Monitoring acompanha drift de dados e de modelo — e aqui criamos o monitor de inferência com uma métrica de negócio customizada: % de clientes em alto risco de churn."

In [0]:
# 7a. Baseline do monitor: split train scoreado pelo Champion + rótulo + timestamp de referência
import pandas as pd

baseline_table_name = f"{catalog}.{schema}.churn_monitor_baseline"

train_pdf = spark.sql(f"""
SELECT c.customer_id, f.tenure_months, f.contract_type, f.monthly_charges, f.total_charges,
       f.payment_method, f.internet_service, f.phone_service, f.paperless_billing, f.partner,
       f.dependents, f.senior_citizen, f.num_optional_services, f.total_tickets, f.billing_tickets,
       f.technical_tickets, f.cancellation_tickets, f.high_priority_tickets, f.avg_resolution_hours,
       {catalog}.{schema}.avg_price_increase(f.monthly_charges, f.tenure_months) AS avg_price_increase,
       c.churn
FROM {labels_table} c
JOIN {feature_table_name} f ON c.customer_id = f.customer_id
WHERE c.split = 'train'
""").toPandas()

X_base = train_pdf[input_cols]
train_pdf["prediction"] = batch_model.predict(X_base).astype("float64")
train_pdf["churn_probability"] = batch_model.predict_proba(X_base)[:, 1]
train_pdf["churn_label"] = (train_pdf["churn"] == "Yes").astype("float64")
train_pdf["inference_timestamp"] = pd.Timestamp("2026-11-09", tz="UTC")  # referência anterior às janelas

from pyspark.sql import functions as F

base_cols = ["customer_id", "inference_timestamp"] + input_cols + ["prediction", "churn_probability", "churn_label"]
# O monitor InferenceLog exige probabilidade em coluna MAP e precisa de model_id no baseline
base_sdf = (spark.createDataFrame(train_pdf[base_cols])
    .withColumn("model_id", F.lit(f"churn_model_v{champion_version}"))
    .withColumn("probability_map", F.create_map(
        F.lit("0"), F.lit(1.0) - F.col("churn_probability"),
        F.lit("1"), F.col("churn_probability"))))
base_sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(baseline_table_name)
spark.sql(f"ALTER TABLE {baseline_table_name} SET TBLPROPERTIES ('comment' = 'Baseline do monitor de churn — train scoreado pelo Champion')")

print(f"{baseline_table_name}: {spark.table(baseline_table_name).count():,} linhas (baseline)")

In [0]:
# 7b. Reescreve churn_predictions com metadados de monitoramento: model_id, churn_label e duas janelas diárias
ts_w1 = pd.Timestamp("2026-11-10", tz="UTC")
ts_w2 = pd.Timestamp("2026-11-11", tz="UTC")

batch_pdf["model_id"] = f"churn_model_v{champion_version}"
batch_pdf["churn_label"] = (batch_pdf["churn"] == "Yes").astype("float64")
half = len(batch_pdf) // 2
batch_pdf["inference_timestamp"] = [ts_w1] * half + [ts_w2] * (len(batch_pdf) - half)

monitor_cols = ["customer_id", "model_id", "inference_timestamp"] + input_cols + ["prediction", "churn_probability", "churn_label"]
# probability_map (MAP<string,double>) é a coluna exigida pelo monitor p/ prediction_proba_col
from pyspark.sql import functions as F
pred_sdf = (spark.createDataFrame(batch_pdf[monitor_cols])
    .withColumn("probability_map", F.create_map(
        F.lit("0"), F.lit(1.0) - F.col("churn_probability"),
        F.lit("1"), F.col("churn_probability"))))
pred_sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(predictions_table_name)
spark.sql(f"ALTER TABLE {predictions_table_name} SET TBLPROPERTIES ('comment' = 'Predições de churn — modelo Champion, com metadados p/ Lakehouse Monitoring')")

print(f"{predictions_table_name} reescrita com metadados: {spark.table(predictions_table_name).count():,} linhas")
display(spark.sql(f"""
SELECT model_id, DATE(inference_timestamp) AS janela, COUNT(*) AS n,
       ROUND(AVG(churn_probability), 3) AS prob_media, ROUND(AVG(churn_label), 3) AS churn_realizado
FROM {predictions_table_name} GROUP BY 1, 2 ORDER BY 2
"""))

In [0]:
# 7c. Cria o monitor Lakehouse Monitoring (InferenceLog + métrica customizada % alto risco)
import time
from databricks.sdk.errors import NotFound
from databricks.sdk.service.catalog import (
    MonitorInferenceLog, MonitorInferenceLogProblemType,
    MonitorMetric, MonitorMetricType, MonitorInfoStatus,
)

# Volume p/ os ativos do monitor (dashboards e tabelas de métricas)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.churn_monitor_assets")

monitor_created = False
monitor_info = None
try:
    try:
        w.quality_monitors.delete(table_name=predictions_table_name)
        print("Monitor anterior removido (idempotência)...")
    except NotFound:
        pass

    high_risk_pct = MonitorMetric(
        name="high_risk_percentage",
        definition="CAST(SUM(CASE WHEN churn_probability > 0.8 THEN 1.0 ELSE 0.0 END) AS DOUBLE) / CAST(COUNT(*) AS DOUBLE)",
        input_columns=["churn_probability"],
        output_data_type="double",
        type=MonitorMetricType.CUSTOM_METRIC_TYPE_AGGREGATE,
    )

    monitor_info = w.quality_monitors.create(
        table_name=predictions_table_name,
        output_schema_name=f"{catalog}.{schema}",
        assets_dir=f"/Volumes/{catalog}/{schema}/churn_monitor_assets",
        baseline_table_name=baseline_table_name,
        inference_log=MonitorInferenceLog(
            problem_type=MonitorInferenceLogProblemType.PROBLEM_TYPE_CLASSIFICATION,
            timestamp_col="inference_timestamp",
            granularities=["1 day"],
            prediction_col="prediction",
            prediction_proba_col="probability_map",
            model_id_col="model_id",
            label_col="churn_label",
        ),
        custom_metrics=[high_risk_pct],
    )

    # Aguarda ficar ACTIVE (até 20 min)
    deadline = time.time() + 20 * 60
    while time.time() < deadline:
        monitor_info = w.quality_monitors.get(table_name=predictions_table_name)
        if monitor_info.status == MonitorInfoStatus.MONITOR_STATUS_ACTIVE:
            monitor_created = True
            break
        if monitor_info.status in (MonitorInfoStatus.MONITOR_STATUS_ERROR, MonitorInfoStatus.MONITOR_STATUS_FAILED):
            raise RuntimeError(f"Monitor falhou: {monitor_info.latest_monitor_failure_msg}")
        time.sleep(20)

    if monitor_created:
        print(f"Monitor ATIVO em {predictions_table_name}")
        print(f"Dashboard: {w.config.host}/sql/dashboardsv3/{monitor_info.dashboard_id}")
        print(f"Métricas de drift: {monitor_info.drift_metrics_table_name}")
    else:
        print("Monitor não ficou ATIVO no prazo — usando fallback manual.")
except Exception as e:
    print(f"⚠️ Lakehouse Monitoring indisponível nesta workspace: {type(e).__name__}: {str(e)[:300]}")
    print("Fallback: a célula seguinte calcula o drift manualmente (PSI/KS/chi2).")

In [0]:
# 7d. Fallback manual: drift e qualidade por janela (PSI/KS/chi2) — roda se o monitor não foi criado
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

if monitor_created:
    print("Monitor Lakehouse Monitoring ATIVO — drift monitorado automaticamente no dashboard acima.")
    print("O alerta de negócio equivalente seria um SQL Alert sobre a query abaixo:\n")
    print(f"""-- Alerta: % de clientes em alto risco acima de 50%
SELECT DATE(inference_timestamp) AS janela,
       AVG(CASE WHEN churn_probability > 0.8 THEN 1.0 ELSE 0.0 END) AS pct_alto_risco,
       AVG(churn_label) AS churn_realizado
FROM {predictions_table_name}
GROUP BY 1
HAVING pct_alto_risco > 0.5""")
else:
    base = train_pdf  # distribuição de referência (train scoreado)
    w1 = batch_pdf[batch_pdf["inference_timestamp"] == ts_w1]
    w2 = batch_pdf[batch_pdf["inference_timestamp"] == ts_w2]

    def psi(expected, actual, bins=10):
        qs = np.quantile(expected, np.linspace(0, 1, bins + 1))
        qs[0], qs[-1] = -np.inf, np.inf
        e = np.histogram(expected, bins=qs)[0] / len(expected)
        a = np.histogram(actual, bins=qs)[0] / len(actual)
        e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
        return float(np.sum((a - e) * np.log(a / e)))

    drift_feats = ["tenure_months", "monthly_charges", "high_priority_tickets", "billing_tickets", "avg_price_increase", "churn_probability"]
    rows = []
    for feat in drift_feats:
        ks = stats.ks_2samp(base[feat].astype(float), w2[feat].astype(float))
        rows.append({"feature": feat, "psi_baseline_w2": psi(base[feat].astype(float), w2[feat].astype(float)),
                     "psi_w1_w2": psi(w1[feat].astype(float), w2[feat].astype(float)),
                     "ks_stat": round(float(ks.statistic), 4), "ks_pvalue": round(float(ks.pvalue), 4)})

    drift_pdf = pd.DataFrame(rows)
    drift_pdf["drift_significativo"] = drift_pdf["psi_baseline_w2"] > 0.2  # PSI > 0.2 = mudança grande
    display(drift_pdf)

    # Chi2 p/ categórica mais importante (contract_type): janela 1 vs janela 2
    ct = pd.crosstab(pd.concat([w1["contract_type"], w2["contract_type"]]),
                     pd.concat([pd.Series(["w1"] * len(w1), index=w1.index), pd.Series(["w2"] * len(w2), index=w2.index)]))
    chi2, p, _, _ = stats.chi2_contingency(ct)
    print(f"contract_type — chi2 entre janelas: {chi2:.2f} (p={p:.4f})")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(drift_pdf["feature"], drift_pdf["psi_baseline_w2"], color="steelblue")
    ax.axhline(0.2, color="red", linestyle="--", label="limiar PSI 0.2")
    ax.set_title("Drift por feature (PSI baseline vs janela 11/11)")
    ax.set_ylabel("PSI")
    ax.legend()
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    print(f"""-- Alerta: % de clientes em alto risco acima de 50%
SELECT DATE(inference_timestamp) AS janela,
       AVG(CASE WHEN churn_probability > 0.8 THEN 1.0 ELSE 0.0 END) AS pct_alto_risco,
       AVG(churn_label) AS churn_realizado
FROM {predictions_table_name}
GROUP BY 1
HAVING pct_alto_risco > 0.5""")

### Por que Lakehouse Monitoring (e não terceiro ou monitore próprio)?
* **Integrado ao Lakehouse**: o monitor nasce dentro da plataforma — acesso, governança e Unity Catalog pelos quais os dados já passam.
* **Métricas em tabelas Delta**: perfil, drift e estatísticas de confusão são gravados como tabelas no seu schema — auditáveis e consultáveis por SQL, não caixa-preta.
* **Sem ferramenta extra para licenciar**: não há necessidade de comprar um monitoramento de ML de terceiros para acompanhar drift e qualidade de modelo.
* **Sem pipeline custom**: não precisamos construir um sistema de monitoramento do zero — o monitor agenda as computações e ainda gera um dashboard pronto; SQL Alerts fecham o ciclo de ação.

## 8. FECHAMENTO
*Talk track: "Recapitulando o ciclo completo: dados sintéticos, feature table com point-in-time, treino com Optuna + MLflow, registro com aliases Champion/Challenger, serving em tempo real, inferência em batch e monitoramento de drift — tudo dentro do Databricks."

In [0]:
# 8a. Resumo do ciclo: passo a passo -> componente Databricks usado
import pandas as pd

resumo = pd.DataFrame([
    {"Passo": "1. Dados sintéticos", "Componente": "Spark (PySpark) + Delta Lake — tabelas customers, service_tickets, churn_labels"},
    {"Passo": "2. Exploração", "Componente": "SQL no Spark (spark.sql) — distribuições e correlações"},
    {"Passo": "3. Features", "Componente": "Feature Store (FeatureEngineeringClient) + função Python UC avg_price_increase"},
    {"Passo": "4. Treino", "Componente": "MLflow (runs aninhados, log_model) + Optuna (10 trials) + LightGBM + SHAP"},
    {"Passo": "Registro", "Componente": "Model Registry do Unity Catalog — aliases Champion/Challenger + tags"},
    {"Passo": "5. Tempo real", "Componente": "Model Serving — endpoint churn-model-recife-dug-simple"},
    {"Passo": "6. Batch", "Componente": "mlflow.sklearn.load_model(models:/@Champion) + tabela churn_predictions"},
    {"Passo": "7. Monitoramento", "Componente": "Lakehouse Monitoring — monitor InferenceLog + métrica customizada + drift"},
    {"Passo": "8. Governança", "Componente": "Unity Catalog — tudo em workspace.recife_dug_simple, com comentários"},
])
display(resumo)

In [0]:
# 8b. Checklist dos objetos criados (verificação real de existência, não só print)
from mlflow import MlflowClient

checks = []

def check(nome, fn):
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    checks.append({"objeto": nome, "existe": "✅" if ok else "❌"})

# Tabelas
for t in ["customers", "service_tickets", "churn_labels", "churn_feature_table", "churn_predictions", "churn_monitor_baseline"]:
    check(f"{catalog}.{schema}.{t}", lambda t=t: spark.catalog.tableExists(f"{catalog}.{schema}.{t}"))

# Função UC
check(f"função {catalog}.{schema}.avg_price_increase", lambda: spark.sql(f"DESCRIBE FUNCTION {catalog}.{schema}.avg_price_increase").count() > 0)

# Volume do monitor
check(f"volume {catalog}.{schema}.churn_monitor_assets", lambda: spark.sql(f"DESCRIBE VOLUME {catalog}.{schema}.churn_monitor_assets").count() > 0)

# Modelo registrado + aliases
reg_client = MlflowClient(registry_uri="databricks-uc")
check(f"modelo {model_name} + alias Champion", lambda: reg_client.get_model_version_by_alias(model_name, "Champion").version)
check(f"alias Challenger", lambda: reg_client.get_model_version_by_alias(model_name, "Challenger").version)

# Experimento MLflow
check(f"experimento {experiment_name}", lambda: reg_client.get_experiment_by_name(experiment_name))

# Endpoint de serving
check(f"endpoint {endpoint_name}", lambda: w.serving_endpoints.get(name=endpoint_name))

# Monitor Lakehouse
check(f"monitor em {predictions_table_name}", lambda: w.quality_monitors.get(table_name=predictions_table_name).status)

checklist = pd.DataFrame(checks)
display(checklist)
print(f"{int((checklist['existe'] == '✅').sum())}/{len(checklist)} objetos confirmados")

In [0]:
# 8c. LIMPEZA — NÃO EXECUTAR: célula inteiramente comentada por segurança
# Apaga TODOS os objetos criados por este demo (tabelas, função, volume, monitor, endpoint, modelo).
# Confirmação dupla: descomente TODAS as linhas abaixo E mude CONFIRM para 'DELETE'.

# CONFIRM = 'NO'
# if CONFIRM == 'DELETE':
#     from databricks.sdk import WorkspaceClient
#     from mlflow import MlflowClient
#     w = WorkspaceClient()
#     uc_client = MlflowClient(registry_uri="databricks-uc")
#     # 1) Monitor Lakehouse
#     w.quality_monitors.delete(table_name=f"{catalog}.{schema}.churn_predictions")
#     # 2) Endpoint de Model Serving
#     w.serving_endpoints.delete(name=endpoint_name)
#     # 3) Modelo registrado (versões + aliases)
#     for mv in uc_client.search_model_versions(f"name='{model_name}'"):
#         uc_client.delete_model_version(mv.name, mv.version)
#     uc_client.delete_registered_model(model_name)
#     # 4) Volume de ativos do monitor
#     spark.sql(f"DROP VOLUME IF EXISTS {catalog}.{schema}.churn_monitor_assets")
#     # 5) Tabelas
#     for t in ["churn_predictions", "churn_monitor_baseline", "churn_feature_table", "churn_labels", "service_tickets", "customers"]:
#         spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.{t}")
#     # 6) Função UC
#     spark.sql(f"DROP FUNCTION IF EXISTS {catalog}.{schema}.avg_price_increase")
#     print("Todos os objetos do demo foram apagados.")
# else:
#     print("Nada apagado — CONFIRM != 'DELETE'.")